In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver # checkpoint saved in ram or memory

In [ ]:
load_dotenv()
llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the party? He heard it was going to be a pizza par-tay!',
 'explanation': 'This joke is a play on words, using "pizza par-tay" as a pun for "pizza party." The joke implies that the pizza went to the party because it was told it was going to be a fun, pizza-themed event. It plays on the idea of a party being a "par-tay" (a more informal or fun way to say party) and the fact that pizzas are often a popular choice for party food.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? He heard it was going to be a pizza par-tay!', 'explanation': 'This joke is a play on words, using "pizza par-tay" as a pun for "pizza party." The joke implies that the pizza went to the party because it was told it was going to be a fun, pizza-themed event. It plays on the idea of a party being a "par-tay" (a more informal or fun way to say party) and the fact that pizzas are often a popular choice for party food.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1754ba-a8d6-6e29-8006-c78781dc36bd'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-01T12:52:06.958822+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1754ba-9359-6ece-8005-1925b8cc55e4'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? He heard it was going to be a pizza par-tay!', 'explanation': 'This joke is a play on words, using "pizza par-tay" as a pun for "pizza party." The joke implies that the pizza went to the party because it was told it was going to be a fun, pizza-themed event. It plays on the idea of a party being a "par-tay" (a more informal or fun way to say party) and the fact that pizzas are often a popular choice for party food.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1754ba-a8d6-6e29-8006-c78781dc36bd'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-01T12:52:06.958822+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1754ba-9359-6ece-8005-1925b8cc55e4'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party? He heard it was go

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the party? Because it heard it was going to be a real pasta-bility!',
 'explanation': 'This joke is a play on words using the term "possibility." The pasta went to the party because it heard it was going to be a real "pasta-bility," which sounds like "possibility." This joke is funny because it uses a pun to suggest that the pasta went to the party because it thought there was a chance it would be a great time.'}

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party? Because it heard it was going to be a real pasta-bility!', 'explanation': 'This joke is a play on words using the term "possibility." The pasta went to the party because it heard it was going to be a real "pasta-bility," which sounds like "possibility." This joke is funny because it uses a pun to suggest that the pasta went to the party because it thought there was a chance it would be a great time.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1754c0-3754-607e-8002-60f71ea9c57b'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-01T12:54:36.117599+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1754c0-2c7c-67d5-8001-1e3082e92add'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party? Because it heard it was going to b